## Interactive agent: Receives a query from the user and responds based on available tools and resources.

The interactive agent is made up of 2 agents:

- The receipent agent receives the request and augments the request (enriches) to provide as many relevent information.
- The processing agent reviews the modified query and compiles feedback that will be sent to an agent.

import libraries 

In [ ]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner,trace, function_tool
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from openai import AsyncOpenAI
import requests
import asyncio
import httpx
from typing import Any
import os 
load_dotenv(override=True)

True

Create a client for the anthropic's API

In [3]:
anthropic_client = AsyncOpenAI(
    api_key= os.getenv(key='ANTHROPIC_API_KEY'),
    base_url='https://api.anthropic.com/v1/',
)

create model

In [4]:
anthropic_model = OpenAIChatCompletionsModel(
            model="claude-haiku-4-5",
            openai_client=anthropic_client,
            )
openai_model = "gpt-4o-mini"

### Recipient agent

In [5]:
system_prompt = """You are an AI assistant whose task is to analyze and rewrite a user’s original query into a clear, detailed, and easy-to-understand instruction that another AI agent can directly execute.
Your rewritten version should:

- Preserve the original intent and requirements of the user.
- Eliminate ambiguity, vague wording, or implicit assumptions.
- Explicitly state what needs to be done, what the expected output is, and any constraints or preferences implied in the original query.
- Be written in plain, unambiguous language that is easy for another agent to interpret and act upon without further clarification.
- Avoid adding new requirements or changing the original meaning—your role is clarification, not expansion of scope.

The goal is to transform the user’s request into an action-ready instruction that optimizes accuracy and execution by the next agent in the workflow.
DO NOT include any explanations, just provide the rewritten instruction in your response.

Then handoff the rewritten instruction to the next agent for further processing and execution."""

In [6]:
ai_assistent = Agent(name="AI assistant",
                      instructions=system_prompt,
                      model= openai_model,
                      )

In [7]:
message = "I want to know how to make a cake, but I don't have much time. Can you give me a quick recipe that I can follow easily?"

In [8]:
with trace("rewriting agent") as t:
    rewritten_message = await Runner.run(ai_assistent,message)

In [9]:
print(rewritten_message.final_output)

Provide a quick and easy cake recipe that can be prepared in a short amount of time. Ensure the recipe includes a list of ingredients, step-by-step instructions, and any necessary baking times and temperatures. Focus on a simple flavor, such as vanilla or chocolate, to facilitate ease of preparation.


convert the agent into a tool

In [10]:
recipient_tool = ai_assistent.as_tool(tool_name="rewrite_tool", tool_description="an AI assistant that rewrites user input into easily processable output for downstream agents")

Fetch data tool

In [ ]:
ALL_ENDPOINTS = ["/data/exception", "/data/logic", "/data/dictionary", "/data/summary"]

In [14]:
async def _fetch_one(client: httpx.AsyncClient, endpoint: str) -> tuple[str, Any]:
    BASE_URL = "https://controldev-apfxc7h7etf4breb.southafricanorth-01.azurewebsites.net"
    key = endpoint.split("/")[-1]
    try:
        response = await client.get(BASE_URL + endpoint)
        if response.status_code == 200:
            return key, response.json()
    except httpx.RequestError:
        pass
    return key, None

@function_tool
async def fetch_control_data(endpoints: list[str] | None = None) -> dict[str, Any]:
    """
    Fetch ControlWeb data from one or more endpoints concurrently.

    Args:
        endpoints: Subset of ['/data/exception', '/data/logic', '/data/dictionary'].
                   Defaults to all three if not provided.

    Returns:
        Dict keyed by endpoint name ('exception', 'logic', 'dictionary').
        Keys for failed/non-200 requests are omitted.
    """
    targets = endpoints if endpoints is not None else ALL_ENDPOINTS
    async with httpx.AsyncClient() as client:
        tasks = [_fetch_one(client, ep) for ep in targets]
        results = await asyncio.gather(*tasks)
    return {key: data for key, data in results if data is not None}

In [15]:
tools = [fetch_control_data]

### Processing agent

In [47]:
ALL_ENDPOINTS = ["/data/exception", "/data/logic", "/data/dictionary", "/data/summary"]

In [48]:
system_prompt2 = f"""You are an AI assistant that handles incoming requests received via a handoff from an agent and responds using the tools available to you.

                        ## Available Tools

                        **fetch_control_data** — Retrieves control information from a database.
                        - **Input:** A list of endpoints (available endpoints: {ALL_ENDPOINTS})
                        - **Output:** Control logic, control exceptions, control summary, and control dictionary

                        ## Behavior Guidelines

                        1. **If the request requires control data:** Call `fetch_control_data` with the appropriate endpoint(s) and use the returned data to construct an accurate, relevant response.

                        2. **If the request is outside your capabilities:** Politely inform the user that you are unable to assist, and briefly explain why (e.g., the request does not map to any available tool or endpoint).

                        ## Principles
                        - Only provide information supported by the available tools and data.
                        - Never fabricate or infer data not returned by a tool.
                        - Keep responses accurate, concise, and directly relevant to the user's query.
                   """
                

Create the processing agent

In [49]:
ai_processing = Agent(name="AI processing agent",
                      instructions=system_prompt2,
                      tools=tools,
                      model= anthropic_model,
                      )

In [50]:
message = "How many exceptions are there?"

In [51]:
print(message)

How many exceptions are there?


In [52]:
handoff=ai_processing

In [53]:
ai_assistent = Agent(name="AI assistant",
                      instructions=system_prompt,
                      model= openai_model,
                      handoffs=[handoff],
                      handoff_description="Pass the rewritten instruction to the AI processing agent for execution.",
                      )

In [54]:
with trace("handoff agent") as t:
    rewritten_message = await Runner.run(ai_assistent,message)

In [55]:
print(rewritten_message.final_output)

There are **2 exceptions** in the system:

1. **Charles Hernandez** (USR010) - Status: Suspended
2. **Isaac Harris** (USR042) - Status: Suspended

Both exceptions were detected on 2026-04-02 and have suspended account statuses.
